## What this notebook does, and what changed (v2)

**Goal.** Measure whether a pipeline around a small local model (Qwen2.5-Coder-7B) solves more BigCodeBench-Hard tasks than a single model call. Final scores always come from the official hidden tests; the pipeline never sees them.

**Systems run here**
- **A0 (baseline):** one model call per task.
- **A3 (full pipeline):** builder (writes a spec) -> planner (writes a plan) -> code generation -> model-written tests run in a sandbox -> up to 2 repair rounds.

**Changes in this version**
1. **Structured spec builder.** The spec is now a checklist with fixed headings: INPUTS, RETURNS, RAISES, BOUNDARIES, REQUIRED LIBRARIES, SIDE EFFECTS, EDGE CASES. It exists to capture details the model missed in the first run: wrong return type, missing `raise`, off-by-one ranges, skipped items.
2. **"Follow the docstring literally" rule** added to the code and repair prompts (exact return type, exact exceptions and conditions, exact boundaries, named libraries only). The planner is told to respect the spec's RETURNS, RAISES and BOUNDARIES. A0 uses the same code prompt, so the comparison stays fair.
3. **Full benchmark:** `SPLIT = "all"`, `N_TASKS = None` (148 tasks) and `SYSTEMS_TO_RUN = ["A0", "A3"]`.
4. `prompt_version = "q7b_v2"` so old v1 result files are not reused, spec token limit raised to 640, and a new last cell prints why failed tasks failed.

**Run time.** A0 takes a few minutes. A3 takes roughly 2 min per task, so several hours in total. Cell 9 is resumable: if the session stops, rerun it and finished tasks are skipped. Save results from `/kaggle/working/results` before the session ends.

# BigCodeBench-Hard: baseline vs pipeline (builder + planner) - LOCAL Qwen2.5-Coder-7B
One self-contained notebook. No GitHub, no repo, no `pip install -e`.

**What it compares** (same model, same tasks; only the front of the pipeline changes)
- **A0** one model call
- **A1** + independent generated tests + sandbox + repair loop
- **A2** A1 + builder (spec)
- **A3** A2 + planner

**Model and settings (aligned with the teammate's HumanEval notebook)**
- `Qwen/Qwen2.5-Coder-7B-Instruct`, run locally on the Kaggle GPU (no API, no rate limit)
- greedy decoding (`do_sample=False`), fp16 (or 4-bit with `USE_4BIT = True`)
- 3 code attempts per task = first try + 2 repairs (her `MAX_ITER = 3`)
- tests written by the model from the task text only; hidden tests are used ONLY for scoring

**How to use:** Settings: Accelerator = **GPU T4 x2**, Internet ON. Run cells top to bottom.
After cell 1 (install) do **Run > Restart session** once, then run from cell 2.
This version runs the FULL BigCodeBench-Hard set (148 tasks) for A0 and A3 only. For a quick trial set `SPLIT = "dev"` and `N_TASKS = 5` in cell 2.

In [1]:
# 1. INSTALL (light). Do NOT pip install the full bigcodebench: it pulls vllm.
!pip install -q --no-deps bigcodebench
!pip install -q appdirs datasets fire gradio_client multipledispatch pqdm rich termcolor tree-sitter tree-sitter-python wget tempdir e2b httpx accelerate bitsandbytes
print("Installed. Now: Run > Restart session, then continue from cell 2.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigcodebench 0.2.5 requires anthropic>=0.26.1, which is not installed.
bigcodebench 0.2.5 requires mistralai<1.0.0,>=0.2.0, which is not installed.
bigcodebench 0.2.5 requires vllm, which is not installed.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
Installed. Now: Run > Restart session, then continue from cell 2.


In [2]:
# 2. SETTINGS
import os, json, re, ast, sys, time, random, math, subprocess, tempfile, threading, collections
from dataclasses import dataclass, field, asdict
from concurrent.futures import ThreadPoolExecutor, as_completed

MODEL      = "Qwen/Qwen2.5-Coder-7B-Instruct"      # local model, same for ALL systems
USE_4BIT   = False                                  # fp16 fits on "GPU T4 x2". Set True on a single 16 GB GPU (P100 / one T4)
SYSTEMS_TO_RUN = ["A0", "A3"]                      # A0 = baseline (1 call), A3 = full pipeline (spec + plan + tests + repair loop)
SPLIT      = "all"                                  # "all" = every BigCodeBench-Hard task (148) | "dev" (20 tasks) | "test" (128 tasks)
N_TASKS    = None                                   # None = all tasks of the split. Use 5 for a quick trial run
WORKERS    = 1                                      # one local GPU = one generation at a time. Keep 1.
MAX_TRIES  = 3                                      # same as the teammate: 3 code attempts = first try + 2 repairs
OUT_DIR    = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
SUBSET, PROMPT_SPLIT = "hard", "complete"          # BigCodeBench-Hard, docstring prompt
EXECUTION  = "gradio"                               # scoring backend hosted on Hugging Face
DEV_SIZE, SPLIT_SEED = 20, 1234

@dataclass
class Config:
    model: str = MODEL
    temperature: float = 0.0                        # greedy decoding (do_sample=False), as in her notebook
    max_tokens: int = 768                           # code / repair calls (her builder uses 768)
    max_iters: int = MAX_TRIES - 1                  # number of REPAIRS after the first try
    workers: int = WORKERS
    prompt_version: str = "q7b_v2"                  # v2 = structured spec + "follow the docstring literally" rule (new result files, old v1 results are not reused)
    stage_max_tokens: dict = field(default_factory=lambda: {"spec": 640, "plan": 512, "tests": 400})
    sandbox: dict = field(default_factory=lambda: {"mem_mb": 8192, "cpu_s": 60, "load_s": 30,
                                                   "test_s": 20, "timeout_s": 120})
cfg = Config()

In [3]:
PROMPTS = {
  # --- v2 change: "follow the docstring literally" rule, shared by the code and the repair prompt ---
  "system_code": "You are an expert Python programmer. Reply with exactly ONE ```python code block containing all imports, any helper functions from the task, and the complete function with the exact name and signature given in the task. Do NOT write docstrings, comments, example usage or test code. No explanation outside the code block.\n\nFOLLOW THE DOCSTRING LITERALLY:\n- Return exactly the type and structure the docstring documents (a tuple is not an array; plain strings are not objects like IPv4Address; use the stated DataFrame columns, order and dtypes).\n- Raise exactly the exceptions listed under 'Raises', under exactly the stated conditions, and validate inputs BEFORE using them (check first, then look up).\n- Respect boundary conventions: inclusive vs exclusive ends, exact number of items, and never silently skip or drop elements the task asks you to process.\n- Use the libraries and functions the task names; do not substitute others.\n- Do not add behaviour the docstring does not ask for.",

  # --- v2 change: the spec is now a structured checklist that extracts the details the model tends to miss ---
  "spec_builder": "You analyze a coding task and write a short checklist spec that another programmer will follow literally. No code. Use exactly these headings, one short line per item, and write 'none' when a heading does not apply:\nINPUTS: parameter names, types, defaults.\nRETURNS: the exact return type and structure (e.g. a tuple of (A, B); a DataFrame with columns ...; a list of str) and what each element is.\nRAISES: every exception type and the exact condition that triggers it, as written in the task.\nBOUNDARIES: inclusive or exclusive range ends, how many items are produced, ordering, default values.\nREQUIRED LIBRARIES: modules and functions the task names and that must be used.\nSIDE EFFECTS: plots, files, random seeds, printing.\nEDGE CASES: 3 to 5 cases to handle.\nCopy requirements from the task text. Do not invent requirements.",

  "planner": "You are a senior engineer planning an implementation. Given the task (and an optional spec), write a short numbered plan of 3 to 7 steps: which libraries or functions to use, the order of operations, how to handle edge cases and errors, and what the function returns. If a spec is given, the plan must respect its RETURNS, RAISES and BOUNDARIES lines. One line per step. Plain text only, no code.",

  "fix": "You are an expert Python programmer fixing code. Reply with exactly ONE ```python code block containing the complete corrected code (imports, helpers, function), no docstrings, no comments, no explanation. Keep following the task text literally: exact return type and structure, exact exceptions and their conditions, exact boundaries, and the libraries the task names.",

  "tests": "You write tests from a task description ONLY. Output ONE ```python code block with 8-10 single-line `assert` statements that call the function (do NOT define the function). Cover normal and edge cases. Only assert behavior the description clearly specifies."
}

In [4]:
# 3. TASKS, TRACE, SANDBOX
import resource  # Linux only
import ast
import re
from dataclasses import dataclass, field


@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str                      # what the generator sees (never the hidden tests)
    entry_point: str
    trusted_tests: list = field(default_factory=list)   # certainly-correct tests taken from the task text
    extra: dict = field(default_factory=dict)


def extract_code(text: str) -> str:
    """Take the code from a markdown block, even when the closing fence is missing."""
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return m.group(1).strip()
    return text.strip()


def syntax_error(code: str) -> str | None:
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"
    except Exception as e:  # e.g. null bytes
        return f"{type(e).__name__}: {e}"


import time


class Trace:
    def __init__(self, task_id: str, system: str):
        self.task_id = task_id
        self.system = system
        self.events: list[dict] = []

    def _add(self, etype: str, stage: str, **data) -> str:
        event_id = f"e{len(self.events) + 1:03d}"
        self.events.append({"event_id": event_id, "type": etype, "stage": stage,
                            "ts": round(time.time(), 3), **data})
        return event_id

    def llm(self, stage, system, user, output, tokens, latency, finish_reason) -> str:
        return self._add("llm_call", stage, system=system, user=user, output=output,
                         tokens=tokens, latency=latency, finish_reason=finish_reason)

    def sandbox(self, stage, round_, summary, code) -> str:
        return self._add("sandbox_run", stage, round=round_, code=code, fatal=summary["fatal"],
                         trusted=f'{summary["trusted_pass"]}/{summary["trusted_total"]}',
                         generated=f'{summary["gen_pass"]}/{summary["gen_total"]}',
                         failures=summary["failures"][:6])

    def decision(self, stage, name, **data) -> str:
        return self._add("decision", stage, name=name, **data)

    def tokens(self) -> int:
        return sum(e.get("tokens", 0) for e in self.events if e["type"] == "llm_call")

    def latency(self) -> float:
        return sum(e.get("latency", 0.0) for e in self.events if e["type"] == "llm_call")

    def llm_calls(self) -> int:
        return sum(1 for e in self.events if e["type"] == "llm_call")


import json
import os
import re
import subprocess
import sys
import tempfile

RUNNER = r'''
import json, sys, ast, signal, resource
L = json.load(open("limits.json"))
resource.setrlimit(resource.RLIMIT_AS, (L["mem_mb"] << 20, L["mem_mb"] << 20))
resource.setrlimit(resource.RLIMIT_CPU, (L["cpu_s"], L["cpu_s"]))

def _alarm(signum, frame):
    raise TimeoutError("timed out")
signal.signal(signal.SIGALRM, _alarm)

def out(obj):
    print("@@RESULT@@" + json.dumps(obj))

ns = {"__name__": "solution"}
try:
    signal.alarm(L["load_s"])
    exec(compile(open("solution.py").read(), "solution.py", "exec"), ns)
    signal.alarm(0)
except BaseException as e:
    line = getattr(e, "lineno", None)
    text = e.msg if isinstance(e, SyntaxError) else str(e)
    out({"fatal": type(e).__name__ + ": " + text[:300] + (" (line " + str(line) + ")" if line else ""), "results": []})
    sys.exit(0)

def expected_vs_actual(src):
    try:
        node = ast.parse(src.strip()).body[0]
        t = node.test
        if isinstance(node, ast.Assert) and isinstance(t, ast.Compare) and len(t.ops) == 1 and isinstance(t.ops[0], ast.Eq):
            got = eval(ast.unparse(t.left), ns)
            return " (got " + repr(got)[:100] + ", expected " + ast.unparse(t.comparators[0])[:100] + ")"
    except BaseException:
        pass
    return ""

results = []
for t in json.load(open("tests.json")):
    ok, msg = True, ""
    try:
        signal.alarm(L["test_s"])
        if t["kind"] == "example":
            got = eval(t["src"], ns)
            want_txt = t["want"].strip()
            try:
                want = eval(want_txt, {}); has = True
            except Exception:
                has = False
            ok = (got == want) if has else (repr(got) == want_txt)
            if not ok:
                msg = t["src"].strip() + " -> got " + repr(got)[:100] + ", expected " + want_txt[:100]
        else:
            exec(t["src"], ns)
        signal.alarm(0)
    except AssertionError:
        signal.alarm(0)
        ok, msg = False, "assertion failed: " + t["src"].strip()[:200] + expected_vs_actual(t["src"])
    except BaseException as e:
        signal.alarm(0)
        ok, msg = False, t["src"].strip()[:200] + " raised " + type(e).__name__ + ": " + str(e)[:150]
    results.append({"kind": t["kind"], "ok": ok, "msg": msg})
out({"fatal": None, "results": results})
'''

DEFAULT_LIMITS = {"mem_mb": 1024, "cpu_s": 20, "load_s": 5, "test_s": 3, "timeout_s": 40}


def run_tests(code: str, tests: list, limits: dict | None = None) -> dict:
    lim = {**DEFAULT_LIMITS, **(limits or {})}
    with tempfile.TemporaryDirectory() as d:
        for name, content in (("solution.py", code), ("tests.json", json.dumps(tests)),
                              ("limits.json", json.dumps(lim)), ("runner.py", RUNNER)):
            with open(os.path.join(d, name), "w") as f:
                f.write(content)
        env = {"PATH": os.environ.get("PATH", ""), "HOME": d, "MPLBACKEND": "Agg"}
        try:
            p = subprocess.run([sys.executable, "runner.py"], capture_output=True, text=True,
                               timeout=lim["timeout_s"], cwd=d, env=env)
        except subprocess.TimeoutExpired:
            return {"fatal": "Timeout (infinite loop?)", "results": []}
    m = re.search(r"@@RESULT@@(.*)", p.stdout)
    if not m:
        return {"fatal": "Process crashed: " + p.stderr[-300:], "results": []}
    return json.loads(m.group(1))


def summarize(ev: dict) -> dict:
    """Turn raw sandbox output into counts + flags.

    trusted_ok / all_ok are NEVER true when there are zero trusted tests / zero tests at all
    (the v2 notebook treated "0 of 0 passed" as success, which accepted unverified code).
    """
    res = ev["results"]
    trusted = [r for r in res if r["kind"] == "example"]
    gen = [r for r in res if r["kind"] == "assert"]
    fatal = ev.get("fatal")
    tp, gp = sum(r["ok"] for r in trusted), sum(r["ok"] for r in gen)
    trusted_ok = fatal is None and len(trusted) > 0 and tp == len(trusted)
    all_ok = (fatal is None and len(res) > 0 and tp == len(trusted) and gp == len(gen))
    return {"fatal": fatal, "trusted_total": len(trusted), "trusted_pass": tp,
            "gen_total": len(gen), "gen_pass": gp, "trusted_ok": trusted_ok, "all_ok": all_ok,
            "n_tests": len(res), "failures": [r for r in res if not r["ok"]],
            "score": (0 if fatal else 1, tp, gp)}

In [5]:
# 4. MODEL CLIENT (local Qwen 7B) + PIPELINE (builder, planner, tests, repair loop)
import time



class LLM:
    """Local Hugging Face model, greedy decoding. Same chat() interface as before, so the pipeline is unchanged."""

    def __init__(self, cfg: Config):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.cfg = cfg
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(cfg.model)
        kw = dict(device_map="auto")
        if USE_4BIT:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        else:
            kw["torch_dtype"] = torch.float16          # T4 does not support bfloat16
        self.model = AutoModelForCausalLM.from_pretrained(cfg.model, **kw).eval()
        self._lock = threading.Lock()
        print("Model loaded on:", self.model.device)

    def chat(self, system: str, user: str, trace: Trace, stage: str, temperature=None) -> str:
        max_new = self.cfg.stage_max_tokens.get(stage.split("_")[0], self.cfg.max_tokens)
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        t0 = time.time()
        with self._lock, self.torch.no_grad():
            inputs = self.tok(text, return_tensors="pt").to(self.model.device)
            out = self.model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                                      pad_token_id=self.tok.eos_token_id)
        gen = out[0][inputs.input_ids.shape[1]:]
        reply = self.tok.decode(gen, skip_special_tokens=True)
        finish = "length" if len(gen) >= max_new else "stop"
        trace.llm(stage, system, user, reply, int(inputs.input_ids.shape[1] + len(gen)),
                  round(time.time() - t0, 2), finish)
        return reply


class MockLLM:
    """responder(stage, system, user) -> str. For tests and dry runs."""

    def __init__(self, responder):
        self.responder = responder

    def chat(self, system: str, user: str, trace: Trace, stage: str, temperature=None) -> str:
        text = self.responder(stage, system, user)
        trace.llm(stage, system, user, text, tokens=len(text) // 4, latency=0.0, finish_reason="stop")
        return text


import ast
from dataclasses import dataclass



@dataclass(frozen=True)
class SystemCfg:
    name: str
    builder: bool = False     # spec step added to the generator prompt
    plan: bool = False        # numbered implementation plan added to the generator prompt
    tests: bool = False       # independent generated tests
    repair: bool = False      # sandbox + repair loop
    guard: bool = False       # fix syntax errors before running anything
    fallback: bool = False    # regenerate cleanly if the code can not be parsed / model is stuck


# ----------------------------------------------------------------------------- helpers
def _gen(llm, system: str, user: str, trace: Trace, stage: str, guard: bool) -> str:
    code = extract_code(llm.chat(system, user, trace, stage))
    if guard:
        for k in range(2):
            err = syntax_error(code)
            if not err:
                break
            fix_user = (f"{user}\n\nYOUR CODE HAD A SYNTAX ERROR: {err}\n\nCODE:\n```python\n{code}\n```\n"
                        "Return the complete corrected code.")
            code = extract_code(llm.chat(system, fix_user, trace, f"{stage}_syntaxfix{k + 1}"))
    return code


def parse_generated_tests(text: str, entry_point: str) -> list:
    block = extract_code(text)
    tests = []
    try:
        for node in ast.parse(block).body:
            if isinstance(node, ast.Assert):
                src = ast.get_source_segment(block, node)
                if src and entry_point in src and "\n" not in src.strip():
                    tests.append({"kind": "assert", "src": src})
    except SyntaxError:
        for line in block.splitlines():
            if line.strip().startswith("assert") and entry_point in line:
                tests.append({"kind": "assert", "src": line.strip()})
    seen, out = set(), []
    for t in tests:
        if t["src"] not in seen:
            seen.add(t["src"])
            out.append(t)
    return out[:10]


def build_feedback(prompt: str, code: str, s: dict) -> str:
    lines = []
    if s["fatal"]:
        lines.append("The program failed to load or run: " + s["fatal"])
    for f in s["failures"][:6]:
        tag = ("[from the task examples: definitely correct]" if f["kind"] == "example"
               else "[generated test: may be wrong]")
        lines.append(tag + " " + f["msg"])
    return ("TASK:\n" + prompt + "\n\nYOUR PREVIOUS CODE:\n```python\n" + code + "\n```\n\nPROBLEMS FOUND:\n"
            + "\n".join(lines) + "\n\nFix the code. Tests marked 'definitely correct' must pass. If a generated "
            "test contradicts the task description, trust the description and ignore that test.")


def _result(task: Task, system: str, cfg: Config, trace: Trace, **fields) -> dict:
    return {"task_id": task.task_id, "dataset": task.dataset, "system": system, "model": cfg.model,
            "prompt_version": cfg.prompt_version, "tokens": trace.tokens(), "latency": round(trace.latency(), 2),
            "llm_calls": trace.llm_calls(), "events": trace.events, **fields}


# ----------------------------------------------------------------------------- systems
def run_baseline(task: Task, llm, prompts: dict, cfg: Config, sys_cfg: SystemCfg) -> dict:
    """A0: one call, same code prompt as the pipeline, no spec, no tests, no guard, no repair."""
    trace = Trace(task.task_id, sys_cfg.name)
    code = _gen(llm, prompts["system_code"], task.prompt, trace, "generate", guard=False)
    return _result(task, sys_cfg.name, cfg, trace, final_code=code, trajectory=[code], candidates=[code],
                   self_pass=None, reason="single_call", verification="none", repairs=0,
                   spec="", plan="", tests=[], history=[])


def run_pipeline(task: Task, llm, prompts: dict, cfg: Config, sys_cfg: SystemCfg) -> dict:
    trace = Trace(task.task_id, sys_cfg.name)
    ep = task.entry_point
    limits = cfg.sandbox

    spec = llm.chat(prompts["spec_builder"], task.prompt, trace, "spec") if sys_cfg.builder else ""
    plan = ""
    if sys_cfg.plan:
        plan_user = task.prompt + (f"\n\nSpec:\n{spec}" if spec else "")
        plan = llm.chat(prompts["planner"], plan_user, trace, "plan")
    gen_tests = (parse_generated_tests(llm.chat(prompts["tests"], task.prompt, trace, "tests"), ep)
                 if sys_cfg.tests else [])
    tests = list(task.trusted_tests) + gen_tests
    has_t, has_g = bool(task.trusted_tests), bool(gen_tests)
    verification = ("trusted+generated" if has_t and has_g else "trusted_only" if has_t
                    else "generated_only" if has_g else "none")

    sys_gen = prompts["system_code"]
    if spec:
        sys_gen += "\n\nTask spec:\n" + spec
    if plan:
        sys_gen += "\n\nImplementation plan (follow it):\n" + plan

    code = _gen(llm, sys_gen, task.prompt, trace, "generate", sys_cfg.guard)
    if sys_cfg.fallback and syntax_error(code):
        trace.decision("generate", "fallback_clean_generation", reason="code still has a syntax error")
        clean = _gen(llm, prompts["system_code"], task.prompt, trace, "generate_fallback", sys_cfg.guard)
        if not syntax_error(clean):
            code = clean

    max_iters = cfg.max_iters if sys_cfg.repair else 0
    best, trajectory, candidates, history = None, [], [], []
    reason, repairs, used_fallback = "max_iters", 0, False

    for it in range(max_iters + 1):
        s = summarize(run_tests(code, tests, limits))
        trace.sandbox(f"sandbox_{it}", it, s, code)
        candidates.append(code)
        if best is None or s["score"] > best["score"]:
            best = {"code": code, "score": s["score"]}
        trajectory.append(best["code"])
        history.append({"round": it, "fatal": s["fatal"], "trusted": f'{s["trusted_pass"]}/{s["trusted_total"]}',
                        "generated": f'{s["gen_pass"]}/{s["gen_total"]}', "best_score": list(best["score"])})
        if s["all_ok"]:
            reason = "all_tests_pass"
            break
        if s["n_tests"] == 0 and not s["fatal"]:
            reason = "no_tests"
            break
        if it == max_iters:
            break

        repairs += 1
        new_code = _gen(llm, prompts["fix"], build_feedback(task.prompt, code, s), trace, f"repair_{it + 1}",
                        sys_cfg.guard)
        if new_code.strip() == code.strip():
            if s["trusted_ok"]:
                reason = "same_code_trusted_ok"
                trace.decision(f"repair_{it + 1}", "same_code_accepted",
                               why="model returned the same code and it passes every trusted test")
                break
            if s["fatal"] and sys_cfg.fallback and not used_fallback:
                used_fallback = True
                trace.decision(f"repair_{it + 1}", "fallback_after_stuck", fatal=s["fatal"])
                new_code = _gen(llm, prompts["system_code"], task.prompt, trace, "generate_fallback", sys_cfg.guard)
                if new_code.strip() != code.strip():
                    code = new_code
                    continue
            reason = "stuck"
            break
        code = new_code

    accepted = reason in ("all_tests_pass", "same_code_trusted_ok")
    return _result(task, sys_cfg.name, cfg, trace, final_code=best["code"], trajectory=trajectory,
                   candidates=candidates, self_pass=accepted, reason=reason, verification=verification,
                   repairs=repairs, spec=spec, plan=plan, tests=tests, history=history)


def run_task(task: Task, llm, prompts: dict, cfg: Config, sys_cfg: SystemCfg) -> dict:
    if not (sys_cfg.tests or sys_cfg.repair or sys_cfg.builder or sys_cfg.plan):
        return run_baseline(task, llm, prompts, cfg, sys_cfg)
    return run_pipeline(task, llm, prompts, cfg, sys_cfg)


SYSTEMS = {
    "A0": SystemCfg("A0"),
    "A1": SystemCfg("A1", tests=True, repair=True, guard=True, fallback=True),
    "A2": SystemCfg("A2", builder=True, tests=True, repair=True, guard=True, fallback=True),
    "A3": SystemCfg("A3", builder=True, plan=True, tests=True, repair=True, guard=True, fallback=True),
    "A3p": SystemCfg("A3p", plan=True, tests=True, repair=True, guard=True, fallback=True),
}

In [6]:
# 5. METRICS + DEV/TEST SPLIT
import collections
import math



def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """95% confidence interval for a proportion."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a) / d, (c + a) / d)


def sign_test(b: int, c: int) -> float:
    """Exact two-sided sign test (= exact McNemar) on discordant pairs b and c."""
    m = b + c
    if m == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(0, min(b, c) + 1)) / 2 ** m)


def summarize_run(records: list, hidden: dict) -> dict:
    """records: result dicts from pipeline.run_task. hidden: {task_id: bool} from hidden scoring."""
    n = len(records)
    passed = sum(1 for r in records if hidden.get(r["task_id"]))
    lo, hi = wilson(passed, n)
    accepted = [r for r in records if r.get("self_pass")]
    rejected = [r for r in records if r.get("self_pass") is False]
    false_accept = [r for r in accepted if not hidden.get(r["task_id"])]
    false_reject = [r for r in rejected if hidden.get(r["task_id"])]

    by_verif = {}
    for v in sorted({r.get("verification") for r in records if r.get("self_pass") is not None}):
        grp = [r for r in records if r.get("verification") == v]
        acc = [r for r in grp if r["self_pass"]]
        by_verif[v] = {"n": len(grp), "accepted": len(acc),
                       "false_accept": sum(1 for r in acc if not hidden.get(r["task_id"])),
                       "pass_rate": round(sum(1 for r in grp if hidden.get(r["task_id"])) / len(grp), 3)}

    final_syntax = sum(1 for r in records if syntax_error(r["final_code"]))
    tokens = sum(r["tokens"] for r in records)
    return {
        "n_tasks": n, "passed": passed, "pass_rate": round(passed / n, 4) if n else 0.0,
        "ci95": [round(lo, 3), round(hi, 3)],
        "avg_tokens": round(tokens / n, 1) if n else 0, "avg_latency_s": round(sum(r["latency"] for r in records) / n, 2) if n else 0,
        "avg_repairs": round(sum(r.get("repairs", 0) for r in records) / n, 3) if n else 0,
        "tokens_per_solved": round(tokens / passed, 1) if passed else None,
        "final_syntax_error_rate": round(final_syntax / n, 3) if n else 0,
        "stop_reasons": dict(collections.Counter(r.get("reason") for r in records)),
        "false_accept": f"{len(false_accept)}/{len(accepted)}",
        "false_reject": f"{len(false_reject)}/{len(rejected)}",
        "by_verification": by_verif,
    }


def compare_runs(hidden_a: dict, hidden_b: dict) -> dict:
    """Paired comparison B vs A on the tasks both runs have."""
    ids = sorted(set(hidden_a) & set(hidden_b))
    fixed = [t for t in ids if hidden_b[t] and not hidden_a[t]]
    broken = [t for t in ids if hidden_a[t] and not hidden_b[t]]
    return {"n_common": len(ids), "fixed_by_B": len(fixed), "broken_by_B": len(broken),
            "sign_test_p": round(sign_test(len(fixed), len(broken)), 4),
            "fixed_ids": fixed, "broken_ids": broken}


import random


def dev_test_split(task_ids: list, dev_size: int = 20, seed: int = 1234) -> tuple[list, list]:
    ids = sorted(task_ids)
    dev = set(random.Random(seed).sample(ids, min(dev_size, len(ids))))
    return [t for t in ids if t in dev], [t for t in ids if t not in dev]


def select(tasks: list, split: str, dev_size: int, seed: int, limit: int | None = None) -> list:
    dev_ids, test_ids = dev_test_split([t.task_id for t in tasks], dev_size, seed)
    wanted = {"dev": set(dev_ids), "test": set(test_ids)}.get(split, {t.task_id for t in tasks})
    chosen = [t for t in tasks if t.task_id in wanted]
    return chosen[:limit] if limit else chosen

In [7]:
# 6. BIGCODEBENCH-HARD: load tasks + hidden scoring (official evaluator)
def load_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, dataset="bigcodebench", prompt=p[key],
                  entry_point=p.get("entry_point", "task_func"), trusted_tests=[],
                  extra={"libs": p.get("libs")})
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

def score(solutions: dict, work_dir: str, calibrated=True) -> dict:
    """{task_id: bool} from the HIDDEN tests. Never used to guide the pipeline."""
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples = os.path.join(work_dir, "samples.jsonl")
    path = samples.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples, "w") as f:
            for tid, c in items.items():
                f.write(json.dumps({"task_id": tid, "solution": c or "pass"}) + "\n")
        if os.path.isfile(path):
            os.remove(path)          # evaluate() reuses an old results file if it finds one

    # 1) ask the server to grade ONLY our tasks: selective_evaluate wants plain numbers "13,15,17"
    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples, execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        # 2) fallback: send a placeholder answer for every task of the subset (slower, always accepted)
        print("selective_evaluate failed (" + str(e)[:200] + ") -> retrying with all tasks padded")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid) for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples, execution=EXECUTION, calibrated=calibrated)
    if not os.path.isfile(path):
        raise RuntimeError("BigCodeBench produced no results file at " + path)
    data = json.load(open(path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

ALL_TASKS = load_tasks()
TASKS = select(ALL_TASKS, SPLIT, DEV_SIZE, SPLIT_SEED, N_TASKS)
print(len(ALL_TASKS), "tasks in BigCodeBench-Hard;", len(TASKS), "selected (split =", SPLIT + ")")

README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/v0.1.0_hf-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  551kB            

data/v0.1.0_hf-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/v0.1.1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  551kB            

data/v0.1.1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/v0.1.2-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  552kB            

... (6 outputs left out)

Generating v0.1.3 split:   0%|          | 0/148 [00:00<?, ? examples/s]

Generating v0.1.4 split:   0%|          | 0/148 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

148 tasks in BigCodeBench-Hard; 148 selected (split = all)


In [8]:
# 7. CHECK THE SCORING FIRST (reference solutions must pass ~100%). If not, the setup is wrong, not the pipeline.
from bigcodebench.data import get_bigcodebench
_probs = get_bigcodebench(subset=SUBSET)
_ids = list(_probs)[:5]
_ref = {i: _probs[i]["complete_prompt"] + "\n" + _probs[i]["canonical_solution"] for i in _ids}
_res = score(_ref, OUT_DIR + "/_check")
print(sum(_res.values()), "/", len(_ids), "reference solutions pass", _res)

Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 1.000
pass@1:	1.000
5 / 5 reference solutions pass {'BigCodeBench/13': True, 'BigCodeBench/15': True, 'BigCodeBench/17': True, 'BigCodeBench/19': True, 'BigCodeBench/34': True}


In [9]:
# 8. SMOKE TEST: A3 on 2 tasks, then read the traces (also loads the model: takes a few minutes the first time)
llm = LLM(cfg)
for t in TASKS[:2]:
    t0 = time.time()
    r = run_task(t, llm, PROMPTS, cfg, SYSTEMS["A3"])
    print(r["task_id"], "| stop:", r["reason"], "| repairs:", r["repairs"], "| verification:", r["verification"],
          "| calls:", r["llm_calls"], "| tokens:", r["tokens"], "| wall time:", round(time.time() - t0), "s")
    print("PLAN:\n" + r["plan"][:600])
    print("finish reasons:", [e["finish_reason"] for e in r["events"] if e["type"] == "llm_call"])
    print("-" * 70)
# finish_reason "length" = the answer was cut off: raise max_tokens (code) in cell 2.
# The wall time above tells you how long the full run will take: multiply by the number of tasks.

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0
BigCodeBench/13 | stop: stuck | repairs: 1 | verification: generated_only | calls: 5 | tokens: 5521 | wall time: 129 s
PLAN:
1. Use `ftplib.FTP` to establish a connection to the FTP server.
2. Attempt to login to the FTP server using the provided credentials.
3. Change to the specified directory on the FTP server.
4. List all files in the current directory.
5. For each file, construct a command string for `wget` to download the file.
6. Execute the `wget` command using `subprocess.run` to download the file.
7. Collect the names of the files that were attempted to be downloaded.
8. Handle exceptions at each step to catch and raise appropriate error messages as specified.
9. Return the list of filenames that were atte
finish reasons: ['stop', 'stop', 'stop', 'stop', 'stop']
----------------------------------------------------------------------
BigCodeBench/15 | stop: stuck | repairs: 2 | verification: generated_only | calls: 6 | tokens: 5582 | wall time: 111 s
PLA

In [10]:
# 9. RUN THE SYSTEMS (resumable: rerun this cell if it stops, finished tasks are kept)
os.makedirs(OUT_DIR, exist_ok=True)
_lock = threading.Lock()

def read_jsonl(path):
    return [json.loads(l) for l in open(path) if l.strip()] if os.path.isfile(path) else []

def run_system(name):
    path = f"{OUT_DIR}/{name}__{SPLIT}__{cfg.prompt_version}.jsonl"
    done = {r["task_id"] for r in read_jsonl(path)}
    todo = [t for t in TASKS if t.task_id not in done]
    print(f"{name}: {len(done)} done, {len(todo)} to run")
    def work(t):
        try:
            return run_task(t, llm, PROMPTS, cfg, SYSTEMS[name])
        except RuntimeError:
            raise                                   # key / credits problem: stop everything
        except Exception as e:                      # one bad task must not kill the run
            return {"task_id": t.task_id, "dataset": "bigcodebench", "system": name, "model": cfg.model,
                    "prompt_version": cfg.prompt_version, "final_code": "", "trajectory": [""], "candidates": [],
                    "self_pass": False if name != "A0" else None, "reason": "error", "verification": "none",
                    "repairs": 0, "spec": "", "plan": "", "tests": [], "history": [], "tokens": 0,
                    "latency": 0.0, "llm_calls": 0, "events": [], "error": f"{type(e).__name__}: {e}"}
    t_start = time.time()
    with ThreadPoolExecutor(cfg.workers) as ex:
        for k, fut in enumerate(as_completed([ex.submit(work, t) for t in todo]), 1):
            r = fut.result()
            with _lock, open(path, "a") as f:
                f.write(json.dumps(r) + "\n")
            el = time.time() - t_start
            print(f"  {name} [{k}/{len(todo)}] {r['task_id']} | calls {r['llm_calls']} | repairs {r.get('repairs')} "
                  f"| elapsed {el/60:.0f} min | ~{el/k*(len(todo)-k)/60:.0f} min left", flush=True)
    keep = {t.task_id for t in TASKS}
    return [r for r in read_jsonl(path) if r["task_id"] in keep]

RECORDS = {name: run_system(name) for name in SYSTEMS_TO_RUN}

A0: 0 done, 148 to run
  A0 [1/148] BigCodeBench/13 | calls 1 | repairs 0 | elapsed 0 min | ~49 min left
  A0 [2/148] BigCodeBench/15 | calls 1 | repairs 0 | elapsed 1 min | ~45 min left
  A0 [3/148] BigCodeBench/17 | calls 1 | repairs 0 | elapsed 1 min | ~37 min left
  A0 [4/148] BigCodeBench/19 | calls 1 | repairs 0 | elapsed 1 min | ~33 min left
  A0 [5/148] BigCodeBench/34 | calls 1 | repairs 0 | elapsed 1 min | ~33 min left
  A0 [6/148] BigCodeBench/37 | calls 1 | repairs 0 | elapsed 1 min | ~33 min left
  A0 [7/148] BigCodeBench/82 | calls 1 | repairs 0 | elapsed 2 min | ~39 min left
  A0 [8/148] BigCodeBench/89 | calls 1 | repairs 0 | elapsed 2 min | ~41 min left
  A0 [9/148] BigCodeBench/92 | calls 1 | repairs 0 | elapsed 3 min | ~40 min left
  A0 [10/148] BigCodeBench/93 | calls 1 | repairs 0 | elapsed 3 min | ~38 min left
  A0 [11/148] BigCodeBench/99 | calls 1 | repairs 0 | elapsed 3 min | ~36 min left
  A0 [12/148] BigCodeBench/100 | calls 1 | repairs 0 | elapsed 3 min | ~3

<unknown>:10: SyntaxWarning: invalid escape sequence '\W'
<unknown>:10: SyntaxWarning: invalid escape sequence '\W'


  A3 [89/148] BigCodeBench/655 | calls 5 | repairs 1 | elapsed 131 min | ~87 min left


<unknown>:10: SyntaxWarning: invalid escape sequence '\W'
<unknown>:6: SyntaxWarning: invalid escape sequence '\W'
<unknown>:6: SyntaxWarning: invalid escape sequence '\W'


  A3 [90/148] BigCodeBench/657 | calls 4 | repairs 0 | elapsed 132 min | ~85 min left
  A3 [91/148] BigCodeBench/678 | calls 4 | repairs 0 | elapsed 134 min | ~84 min left
  A3 [92/148] BigCodeBench/720 | calls 6 | repairs 2 | elapsed 136 min | ~83 min left
  A3 [93/148] BigCodeBench/723 | calls 6 | repairs 2 | elapsed 138 min | ~82 min left
  A3 [94/148] BigCodeBench/752 | calls 4 | repairs 0 | elapsed 140 min | ~80 min left
  A3 [95/148] BigCodeBench/760 | calls 4 | repairs 0 | elapsed 142 min | ~79 min left
  A3 [96/148] BigCodeBench/763 | calls 5 | repairs 1 | elapsed 143 min | ~78 min left
  A3 [97/148] BigCodeBench/765 | calls 4 | repairs 0 | elapsed 145 min | ~76 min left
  A3 [98/148] BigCodeBench/771 | calls 5 | repairs 1 | elapsed 146 min | ~75 min left


<unknown>:9: SyntaxWarning: invalid escape sequence '\['


  A3 [99/148] BigCodeBench/777 | calls 5 | repairs 1 | elapsed 148 min | ~73 min left
  A3 [100/148] BigCodeBench/785 | calls 5 | repairs 1 | elapsed 149 min | ~72 min left
  A3 [101/148] BigCodeBench/800 | calls 5 | repairs 1 | elapsed 151 min | ~70 min left
  A3 [102/148] BigCodeBench/826 | calls 5 | repairs 1 | elapsed 152 min | ~69 min left
  A3 [103/148] BigCodeBench/845 | calls 6 | repairs 2 | elapsed 154 min | ~67 min left
  A3 [104/148] BigCodeBench/854 | calls 4 | repairs 0 | elapsed 155 min | ~66 min left
  A3 [105/148] BigCodeBench/857 | calls 5 | repairs 1 | elapsed 157 min | ~64 min left
  A3 [106/148] BigCodeBench/865 | calls 5 | repairs 1 | elapsed 158 min | ~63 min left
  A3 [107/148] BigCodeBench/870 | calls 4 | repairs 0 | elapsed 159 min | ~61 min left
  A3 [108/148] BigCodeBench/879 | calls 6 | repairs 2 | elapsed 161 min | ~60 min left
  A3 [109/148] BigCodeBench/897 | calls 4 | repairs 0 | elapsed 163 min | ~58 min left
  A3 [110/148] BigCodeBench/906 | calls 5 | 

In [11]:
# 10. SCORE WITH THE HIDDEN TESTS + COMPARE
HIDDEN, SUMMARY = {}, {}
for name, recs in RECORDS.items():
    HIDDEN[name] = score({r["task_id"]: r["final_code"] for r in recs}, f"{OUT_DIR}/score_{name}_{SPLIT}")
    SUMMARY[name] = summarize_run(recs, HIDDEN[name])
    json.dump(HIDDEN[name], open(f"{OUT_DIR}/{name}__{SPLIT}__hidden.json", "w"), indent=1)

import pandas as pd
table = pd.DataFrame({n: {"pass_rate": s["pass_rate"], "95% CI": s["ci95"], "passed": f'{s["passed"]}/{s["n_tasks"]}',
                          "avg_tokens": s["avg_tokens"], "avg_repairs": s["avg_repairs"],
                          "syntax_err_rate": s["final_syntax_error_rate"], "false_accept": s["false_accept"],
                          "false_reject": s["false_reject"], "stop_reasons": s["stop_reasons"]}
                      for n, s in SUMMARY.items()}).T
display(table)

base = SYSTEMS_TO_RUN[0]
for other in SYSTEMS_TO_RUN[1:]:
    c = compare_runs(HIDDEN[base], HIDDEN[other])
    print(f"{other} vs {base}: fixed {c['fixed_by_B']}, broken {c['broken_by_B']}, sign-test p = {c['sign_test_p']}")
# With few tasks, differences are mostly noise. Look at p and the CI, not only the pass rate.

Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.216
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/


<unknown>:11: SyntaxWarning: invalid escape sequence '\s'
<unknown>:10: SyntaxWarning: invalid escape sequence '\W'


BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.182


<unknown>:10: SyntaxWarning: invalid escape sequence '\W'
<unknown>:6: SyntaxWarning: invalid escape sequence '\W'


,pass_rate,95% CI,passed,avg_tokens,avg_repairs,syntax_err_rate,false_accept,false_reject,stop_reasons
A0,0.2162,"[0.158, 0.289]",32/148,767.0,0.0,0.0,0/0,0/0,{'single_call': 148}
A3,0.1824,"[0.129, 0.252]",27/148,4532.1,0.743,0.0,4/6,25/142,"{'stuck': 58, 'all_tests_pass': 6, 'no_tests':..."


A3 vs A0: fixed 10, broken 15, sign-test p = 0.4244


In [12]:
# 11. LOOK AT ONE TASK IN DETAIL (this is what the judge will read later)
name, i = "A3", 0
r = RECORDS[name][i]
print(r["task_id"], "| hidden pass:", HIDDEN[name].get(r["task_id"]), "| stop:", r["reason"], "|", r["verification"])
print("SPEC:\n", r["spec"][:500]); print("PLAN:\n", r["plan"][:500]); print("ROUNDS:", r["history"])

BigCodeBench/13 | hidden pass: False | stop: stuck | generated_only
SPEC:
 **INPUTS:**  
- `ftp_server` (str): The FTP server address. Default is `'ftp.dlptest.com'`.  
- `ftp_user` (str): The FTP server username. Default is `'dlpuser'`.  
- `ftp_password` (str): The FTP server password. Default is `'rNrKYTX9g7z3RgJRmxWuGHbeu'`.  
- `ftp_dir` (str): The directory path on the FTP server from which files need to be downloaded. Default is `'/ftp/test'`.

**RETURNS:**  
- `List[str]`: A list of filenames that were attempted to be downloaded from the FTP server.

**RAISES:*
PLAN:
 1. Use `ftplib.FTP` to establish a connection to the FTP server.
2. Attempt to login to the FTP server using the provided credentials.
3. Change to the specified directory on the FTP server.
4. List all files in the current directory.
5. For each file, construct a command string for `wget` to download the file.
6. Execute the `wget` command using `subprocess.run` to download the file.
7. Collect the names of the f

In [13]:
# 12. WHY DID TASKS FAIL? Hidden-test error messages for failed tasks (first MAX_SHOW only)
import glob
NAME, MAX_SHOW = "A3", 15
paths = glob.glob(f"{OUT_DIR}/score_{NAME}_{SPLIT}/*eval_results.json")
if not paths:
    print("No eval results file found. Run cell 10 first.")
else:
    ev = json.load(open(paths[0]))["eval"]
    failed = [(tid, v[0]) for tid, v in ev.items() if v and v[0]["status"] != "pass"]
    print(len(failed), "failed tasks of", len(ev))
    for tid, v in failed[:MAX_SHOW]:
        print(tid, "FAIL"); print(str(v.get("details"))[:400]); print("-" * 60)

121 failed tasks of 148
BigCodeBench/120 FAIL
{'test_reproducibility_with_seed': 'Traceback (most recent call last):\n  File "__test__.py", line 38, in test_reproducibility_with_seed\nAssertionError: Lists differ: [\'202[5056 chars]11-26\', \'2020-06-02\', \'2020-08-22\', \'2020-06-10\', \'2020-02-07\'] != [\'202[5056 chars]11-26\', \'2020-06-02\', \'2020-08-22\', \'2020-06-10\']\n\nFirst list contains 1 additional elements.\nFirst extra elem
------------------------------------------------------------
BigCodeBench/227 FAIL
{'test_docstring_examples': 'Traceback (most recent call last):\n  File "/usr/local/lib/python3.10/unittest/mock.py", line 1379, in patched\n    return func(*newargs, **newkeywargs)\n  File "__test__.py", line 75, in test_docstring_examples\n  File "__test__.py", line 21, in task_func\nFileNotFoundError: The specified audio file \'audio.wav\' does not exist.\n', 'test_spl_calculation': 'Traceback 
------------------------------------------------------------
BigCodeB